# Módulo 07 · Aula 01 — Consumindo APIs

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Fechamos com a transportadora. Eles têm API: dá para cotar frete e rastrear entrega. Só que ontem a API deles caiu por 8 minutos e o nosso checkout caiu junto. Oito minutos sem vender."*
> — Sua chefe, de novo

E tem mais:

> *"O rastreamento puxa 50 entregas por página. Temos 4.000. O script demora 12 minutos e às vezes trava no meio — aí ninguém sabe se ele terminou ou morreu."*

Duas lições que este módulo inteiro persegue:

| No M06 você era… | Agora você é… |
|------------------|---------------|
| O **servidor** | O **cliente** |
| Quem define o contrato | Quem obedece ao contrato alheio |
| Quem escolhe quando falhar | Quem sofre a falha dos outros |

> 🎯 **A mudança de mentalidade:** como servidor, você controla tudo. Como cliente, você não controla **nada** — nem a disponibilidade, nem a latência, nem o formato que eles vão mudar sem avisar.
>
> Todo o resto desta aula é como programar sob essa premissa.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | `httpx.Client` | Conexão reaproveitada, não descartada |
| 2 | Timeouts | 🔴 O default é o que te derruba |
| 3 | Erros | O que é culpa da rede e o que é da resposta |
| 4 | Autenticação como cliente | Token de terceiro, renovado sozinho |
| 5 | **Retry com backoff** | 🔴 E o que você NUNCA deve repetir |
| 6 | Disjuntor | Parar de bater numa porta fechada |
| 7 | Paginação | Offset vs cursor, e por que importa |
| 8 | `429` e `Retry-After` | Respeitar o limite alheio |

## ⚙️ Como este notebook funciona

Precisamos de uma API de terceiro para consumir. Em vez de depender da internet — que tornaria o notebook frágil e irreprodutível — vamos **subir uma**.

A **Transportadora Veloz** é fictícia, roda em `127.0.0.1` numa porta livre, e é deliberadamente **imperfeita**: ela cai, demora, limita requisições e pagina de dois jeitos diferentes. Como as de verdade.

> 💭 **Por que não o `TestClient` do M06?**
>
> Porque ele roda a aplicação **em processo** — nunca abre um socket. Isso é perfeito para testar a *sua* API, mas aqui o assunto é o oposto: você é o cliente de um serviço alheio.
>
> **Timeout de conexão, recusa de conexão e pool de sockets só existem quando há rede de verdade.** Então há rede de verdade — só que local.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 07
# ═══════════════════════════════════════════════════════════════
import json
import socket
import subprocess
import sys
import threading
import time
import warnings

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("uvicorn[standard]", "uvicorn")]:
    _garantir(_p, _m)

import httpx
import uvicorn
from fastapi import FastAPI

print(f"✅ httpx {httpx.__version__}")


# ═══════════════════════════════════════════════════════════════
#  Servidor de verdade, numa thread
#
#  💭 Por que não o TestClient do M06?
#
#     Porque o TestClient roda a app EM PROCESSO — ele nunca abre um
#     socket. Isso é ótimo para testar a SUA API, mas neste módulo o
#     assunto é o contrário: você é o CLIENTE de um serviço alheio.
#
#     Timeout de conexão, DNS, recusa de conexão e pool de sockets só
#     existem quando há rede de verdade. Então subimos um servidor
#     real em 127.0.0.1 e falamos com ele por HTTP de verdade.
# ═══════════════════════════════════════════════════════════════

def _porta_livre() -> int:
    """Pede ao sistema uma porta que ninguém está usando."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class Servico:
    """Sobe uma app FastAPI num servidor real, em segundo plano.

    Uso:
        with Servico(app) as url:
            httpx.get(f"{url}/ping")

    Ou, para deixar no ar durante várias células:
        servico = Servico(app).iniciar()
        ...
        servico.parar()
    """

    def __init__(self, app: FastAPI, nome: str = "servico"):
        self.app = app
        self.nome = nome
        self.porta = _porta_livre()
        self.url = f"http://127.0.0.1:{self.porta}"
        self._servidor = None
        self._thread = None

    def iniciar(self, timeout: float = 15.0) -> "Servico":
        config = uvicorn.Config(self.app, host="127.0.0.1", port=self.porta,
                                log_level="critical", access_log=False)
        self._servidor = uvicorn.Server(config)
        self._thread = threading.Thread(target=self._servidor.run, daemon=True)
        self._thread.start()

        limite = time.monotonic() + timeout
        while time.monotonic() < limite:
            if self._servidor.started:
                print(f"🟢 {self.nome} no ar em {self.url}")
                return self
            time.sleep(0.05)
        raise RuntimeError(f"{self.nome} não subiu em {timeout}s")

    def parar(self) -> None:
        if self._servidor is not None:
            self._servidor.should_exit = True
            self._thread.join(timeout=10)
            print(f"⚫ {self.nome} desligado")

    def __enter__(self) -> str:
        self.iniciar()
        return self.url

    def __exit__(self, *_):
        self.parar()


# ═══════════════════════════════════════════════════════════════
#  Exibição
# ═══════════════════════════════════════════════════════════════

def mostrar(resposta, rotulo: str = "", corpo: bool = True, linhas: int = 10):
    """Imprime uma resposta httpx de forma legível."""
    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    ms = resposta.elapsed.total_seconds() * 1000
    alvo = rotulo or f"{resposta.request.method} {resposta.request.url.path}"
    print(f"{cor} {alvo:<46} → {resposta.status_code}  ({ms:.0f} ms)")
    if corpo:
        try:
            texto = json.dumps(resposta.json(), ensure_ascii=False, indent=2)
            partes = texto.splitlines()
            for linha in partes[:linhas]:
                print(f"   {linha}")
            if len(partes) > linhas:
                print(f"   ... (+{len(partes) - linhas} linhas)")
        except Exception:
            if resposta.text.strip():
                print(f"   {resposta.text[:200]}")
    return resposta


def cronometrar(funcao, *args, **kwargs):
    """Executa e devolve (resultado, milissegundos)."""
    inicio = time.perf_counter()
    resultado = funcao(*args, **kwargs)
    return resultado, (time.perf_counter() - inicio) * 1000


print("✅ `Servico`, `mostrar()` e `cronometrar()` prontos")

## 0. A Transportadora Veloz

Antes de consumir, vamos construir o que será consumido. Leia o código abaixo com atenção: **cada endpoint existe para reproduzir um problema real**.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  API da Transportadora Veloz — fictícia e deliberadamente imperfeita
# ═══════════════════════════════════════════════════════════════
import base64
import random
import secrets
from datetime import datetime, timedelta, timezone

from fastapi import FastAPI, Header, HTTPException, Query, Request, Response

veloz = FastAPI(title="Transportadora Veloz", version="1.4.2")

CREDENCIAIS = {"aurora-comercio": "s3nh4-da-aurora"}
TOKENS: dict[str, datetime] = {}
ESTADO = {"chamadas": 0, "falhas_restantes": 0}

_ENTREGAS = [
    {"codigo": f"VZ{1000 + i}", "pedido": 9000 + i,
     "cidade": random.Random(i).choice(["Campinas", "São Paulo", "Valinhos",
                                        "Sumaré", "Indaiatuba", "Jundiaí"]),
     "status": random.Random(i).choice(["postado", "em_transito", "entregue"]),
     "peso_kg": round(random.Random(i).uniform(0.3, 12.0), 2)}
    for i in range(137)          # 137: um número que NÃO é múltiplo da página
]


def _conferir_token(authorization: str | None) -> str:
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "token ausente", headers={"WWW-Authenticate": "Bearer"})
    token = authorization.removeprefix("Bearer ")
    expira = TOKENS.get(token)
    if expira is None:
        raise HTTPException(401, "token desconhecido")
    if datetime.now(timezone.utc) > expira:
        raise HTTPException(401, "token expirado")
    return token


# ── Autenticação: OAuth2 client credentials ──
@veloz.post("/v1/auth/token")
def emitir_token(authorization: str | None = Header(default=None)):
    """Autenticação HTTP Basic → devolve um Bearer token de vida curta."""
    if not authorization or not authorization.startswith("Basic "):
        raise HTTPException(401, "credenciais ausentes")
    cru = base64.b64decode(authorization.removeprefix("Basic ")).decode()
    cliente, _, senha = cru.partition(":")
    if CREDENCIAIS.get(cliente) != senha:
        raise HTTPException(401, "credenciais inválidas")

    token = secrets.token_urlsafe(24)
    # ⏱️ 4 segundos: curto de propósito, para você ver a renovação acontecer
    TOKENS[token] = datetime.now(timezone.utc) + timedelta(seconds=4)
    return {"access_token": token, "token_type": "Bearer", "expires_in": 4}


# ── Cotação de frete ──
@veloz.post("/v1/cotacoes")
def cotar(dados: dict, authorization: str | None = Header(default=None)):
    _conferir_token(authorization)
    peso = float(dados.get("peso_kg", 1))
    cep = str(dados.get("cep_destino", ""))
    if len(cep.replace("-", "")) != 8:
        raise HTTPException(422, "cep_destino inválido")
    base = 12.90 + peso * 2.35
    return {"valor": round(base, 2), "prazo_dias": 2 if cep.startswith("13") else 5,
            "servico": "veloz-expresso"}


# ── Instabilidade controlada ──
@veloz.get("/v1/instavel")
def instavel(authorization: str | None = Header(default=None)):
    """Falha as N primeiras vezes, depois funciona.

    Reproduz a falha TRANSITÓRIA — a que vale a pena repetir.
    """
    _conferir_token(authorization)
    ESTADO["chamadas"] += 1
    if ESTADO["falhas_restantes"] > 0:
        ESTADO["falhas_restantes"] -= 1
        raise HTTPException(503, "serviço temporariamente indisponível")
    return {"ok": True, "tentativas_ate_sucesso": ESTADO["chamadas"]}


# ── Lentidão controlada ──
@veloz.get("/v1/lento")
def lento(segundos: float = Query(1.0, ge=0, le=30)):
    time.sleep(segundos)
    return {"dormi_por": segundos}


# ── Limite de taxa ──
_JANELA: dict[str, list[float]] = {}


@veloz.get("/v1/limitado")
def limitado(request: Request, response: Response):
    """No máximo 3 requisições a cada 2 segundos."""
    agora = time.monotonic()
    ip = request.client.host if request.client else "anon"
    recentes = [t for t in _JANELA.get(ip, []) if agora - t < 2.0]
    if len(recentes) >= 3:
        espera = round(2.0 - (agora - recentes[0]), 2)
        raise HTTPException(429, "limite excedido",
                            headers={"Retry-After": str(max(espera, 0.1))})
    recentes.append(agora)
    _JANELA[ip] = recentes
    response.headers["X-RateLimit-Restante"] = str(3 - len(recentes))
    return {"ok": True, "usadas_na_janela": len(recentes)}


# ── Paginação por OFFSET ──
@veloz.get("/v1/entregas")
def listar_entregas(pagina: int = Query(1, ge=1),
                    por_pagina: int = Query(25, ge=1, le=100),
                    authorization: str | None = Header(default=None)):
    _conferir_token(authorization)
    inicio = (pagina - 1) * por_pagina
    fatia = _ENTREGAS[inicio:inicio + por_pagina]
    total_paginas = -(-len(_ENTREGAS) // por_pagina)      # divisão para cima
    return {"pagina": pagina, "por_pagina": por_pagina,
            "total": len(_ENTREGAS), "total_paginas": total_paginas,
            "itens": fatia}


# ── Paginação por CURSOR ──
@veloz.get("/v1/eventos")
def listar_eventos(cursor: str | None = None, limite: int = Query(20, ge=1, le=100),
                   authorization: str | None = Header(default=None)):
    _conferir_token(authorization)
    inicio = int(base64.urlsafe_b64decode(cursor).decode()) if cursor else 0
    fatia = _ENTREGAS[inicio:inicio + limite]
    fim = inicio + len(fatia)
    proximo = (base64.urlsafe_b64encode(str(fim).encode()).decode()
               if fim < len(_ENTREGAS) else None)
    return {"itens": fatia, "proximo_cursor": proximo}


# ── Sobe ──
veloz_servico = Servico(veloz, "Transportadora Veloz").iniciar()
URL = veloz_servico.url

In [ ]:
# Primeira conversa
resposta = httpx.get(f"{URL}/v1/lento?segundos=0")
mostrar(resposta, "GET /v1/lento (sem auth)")

resposta = httpx.get(f"{URL}/v1/entregas")
mostrar(resposta, "GET /v1/entregas (sem token)")

## 1. `Client` — a diferença que quase ninguém mede

`httpx.get(...)` funciona. Mas cada chamada abre uma conexão TCP nova, faz o handshake, transfere e fecha.

Um `Client` mantém um **pool** de conexões abertas.

In [ ]:
N = 30

def sem_cliente():
    for _ in range(N):
        httpx.get(f"{URL}/v1/lento?segundos=0")


def com_cliente():
    with httpx.Client(base_url=URL) as cliente:
        for _ in range(N):
            cliente.get("/v1/lento?segundos=0")


_, ms_sem = cronometrar(sem_cliente)
_, ms_com = cronometrar(com_cliente)

print(f"{N} requisições:\n")
print(f"   httpx.get() solto : {ms_sem:8.1f} ms   ({ms_sem / N:5.2f} ms cada)")
print(f"   com Client        : {ms_com:8.1f} ms   ({ms_com / N:5.2f} ms cada)")
print(f"\n   ganho: {ms_sem / ms_com:.1f}×")

> 🎯 **E isto é em `127.0.0.1`, onde o handshake é quase de graça.**
>
> Contra um servidor na internet, cada conexão nova custa um *round-trip* de TCP mais um *handshake* TLS completo — tipicamente 100–300 ms. Trinta chamadas viram **9 segundos** de puro protocolo.
>
> 🧭 **Regra:** um `Client` por serviço externo, criado uma vez e reaproveitado. Numa API FastAPI, isso significa criá-lo no `lifespan` — não a cada requisição.

In [ ]:
# O Client também guarda a configuração comum
cliente = httpx.Client(
    base_url=URL,
    timeout=httpx.Timeout(5.0, connect=2.0),
    headers={"User-Agent": "atlas-aurora/1.0 (engenharia@aurora.com.br)"},
    follow_redirects=True,
)

r = cliente.get("/v1/lento", params={"segundos": 0})
mostrar(r, "com Client configurado", corpo=False)
print(f"   User-Agent enviado: {r.request.headers['user-agent']}")
print(f"   URL final         : {r.request.url}")

> 💡 **Mande um `User-Agent` que identifique você.** Quando o serviço alheio tiver problema, o suporte deles vai olhar os logs. Um `python-httpx/0.28` anônimo é indistinguível de um bot; `atlas-aurora/1.0 (engenharia@aurora.com.br)` faz alguém conseguir te avisar.
>
> É gentileza — e é interesse próprio.

## 2. 🔴 Timeouts

Este é o item mais importante da aula inteira.

In [ ]:
# O httpx tem um padrão razoável — mas ele é fácil de desligar sem querer
print(f"padrão do httpx        : {httpx.Timeout(5.0)}")
print(f"httpx.Client(timeout=None) : 🔴 espera para SEMPRE\n")

# ── Com timeout: você desiste e segue a vida ──
cliente_curto = httpx.Client(base_url=URL, timeout=1.0)
inicio = time.perf_counter()
try:
    cliente_curto.get("/v1/lento", params={"segundos": 3})
except httpx.ReadTimeout as erro:
    ms = (time.perf_counter() - inicio) * 1000
    print(f"✅ com timeout=1s → desistiu em {ms:.0f} ms")
    print(f"   {type(erro).__name__}: {erro}\n")

# ── Sem timeout: o servidor decide quanto do SEU tempo consumir ──
cliente_sem_limite = httpx.Client(base_url=URL, timeout=None)
inicio = time.perf_counter()
cliente_sem_limite.get("/v1/lento", params={"segundos": 3})
ms = (time.perf_counter() - inicio) * 1000
print(f"🔴 com timeout=None → esperou os {ms:.0f} ms inteiros")
print("   Aqui foram 3 segundos porque nós escolhemos 3.")
print("   Em produção, quem escolhe é o serviço do outro — e pode ser 'para sempre'.")
cliente_sem_limite.close()
cliente_curto.close()

> 🔴 **Sem timeout, uma requisição lenta vira um worker preso para sempre.**
>
> O cenário real: o serviço de terceiro não *cai* — ele fica **lento**. Suas requisições não falham, elas **acumulam**. Com 4 workers, bastam 4 requisições penduradas para a sua API inteira parar de responder. E o monitoramento mostra "0 erros", porque nada errou — só nunca terminou.
>
> 💭 **Esta é a falha em cascata clássica.** Um serviço secundário lento derruba o principal, que derruba quem depende dele. O timeout é o que interrompe a corrente.

In [ ]:
# Os QUATRO timeouts do httpx — cada um mede uma coisa
tempos = httpx.Timeout(
    connect=2.0,   # abrir a conexão TCP  → serviço fora do ar, DNS ruim
    read=5.0,      # esperar cada pedaço  → servidor lento
    write=5.0,     # enviar o corpo       → upload grande, rede ruim
    pool=1.0,      # pegar uma conexão    → SEU pool esgotou
)

cliente = httpx.Client(base_url=URL, timeout=tempos)
print("connect  2s   servidor fora do ar")
print("read     5s   servidor lento")
print("write    5s   upload travando")
print("pool     1s   🔴 SEU pool esgotou — a culpa é sua, não deles")
print()

# `pool` é o mais sutil: ele denuncia problema do SEU lado
r = cliente.get("/v1/lento", params={"segundos": 0})
print(f"✅ com timeouts explícitos: {r.status_code}")

> ⚠️ **`connect` deve ser curto; `read` pode ser mais longo.**
>
> Não conseguir *abrir* a conexão em 2 segundos significa que o serviço está fora do ar — esperar 30 não vai ajudar. Já uma consulta pesada pode legitimamente levar 10 segundos para responder.
>
> 🎯 **O timeout total precisa caber no orçamento de quem te chama.** Se a sua API promete responder em 3 segundos e faz duas chamadas externas de 5 segundos cada, a promessa é falsa. Timeout se planeja de fora para dentro.

## 3. Erros: dois mundos diferentes

In [ ]:
cliente = httpx.Client(base_url=URL, timeout=2.0)

# ── Mundo 1: a requisição CHEGOU, a resposta foi ruim ──
r = cliente.get("/v1/entregas")           # sem token
print(f"1. Resposta recebida: {r.status_code}")
print(f"   r.is_error = {r.is_error}   ← httpx NÃO levanta exceção sozinho\n")

try:
    r.raise_for_status()
except httpx.HTTPStatusError as erro:
    print(f"   raise_for_status() → {type(erro).__name__}")
    print(f"   você ainda tem a resposta: {erro.response.json()}\n")

# ── Mundo 2: a requisição NÃO CHEGOU ──
try:
    httpx.get("http://127.0.0.1:1/nada", timeout=2.0)
except httpx.ConnectError as erro:
    print(f"2. {type(erro).__name__}: não houve resposta nenhuma")
    print("   Não existe status code. Não existe corpo. Nada chegou.")

> 🎯 **A distinção que muda o seu código:**
>
> | | Chegou? | Exceção | O que significa |
> |---|---------|---------|-----------------|
> | `404`, `422`, `500` | ✅ sim | só com `raise_for_status()` | O servidor te ouviu e disse não |
> | `ConnectError` | 🔴 não | sempre | Ninguém atendeu |
> | `ReadTimeout` | 🤷 talvez | sempre | **Você não sabe se foi processado** |
>
> **O `ReadTimeout` é o caso perigoso.** Você mandou "cobre R$ 500 deste cartão" e não recebeu resposta. Foi cobrado? Talvez. Repetir pode cobrar duas vezes; não repetir pode deixar de cobrar.
>
> Guarde esta dúvida — ela volta na seção 5.

In [ ]:
# A hierarquia de exceções do httpx
print("httpx.HTTPError")
print("├── httpx.RequestError          🔴 não houve resposta")
print("│   ├── ConnectError            porta fechada, DNS falhou")
print("│   ├── ConnectTimeout          não abriu a tempo")
print("│   ├── ReadTimeout             ⚠️  não sabe se processou")
print("│   ├── WriteTimeout            não conseguiu enviar")
print("│   ├── PoolTimeout             SEU pool esgotou")
print("│   └── RemoteProtocolError     o servidor falou errado")
print("└── httpx.HTTPStatusError       ✅ houve resposta, e ela é ruim")
print()
print("💡 `except httpx.RequestError` pega TODOS os problemas de rede.")
print("   `except httpx.HTTPError` pega rede E status ruim.")

## 4. Autenticação como cliente

A Veloz usa **client credentials**: você troca usuário e senha por um token de vida curta.

In [ ]:
class ClienteVeloz:
    """Cliente da Transportadora Veloz, com renovação automática de token.

    💭 Por que uma classe? Porque há ESTADO: o token e a validade dele.
       Uma função solta precisaria de uma variável global — e aí dois
       clientes na mesma aplicação brigariam pelo mesmo token.
    """

    def __init__(self, url: str, cliente_id: str, segredo: str):
        self._http = httpx.Client(
            base_url=url,
            timeout=httpx.Timeout(5.0, connect=2.0),
            headers={"User-Agent": "atlas-aurora/1.0"},
        )
        self._id = cliente_id
        self._segredo = segredo
        self._token: str | None = None
        self._expira: datetime = datetime.min.replace(tzinfo=timezone.utc)
        self.renovacoes = 0

    # ── token ──
    def _autenticar(self) -> None:
        credencial = base64.b64encode(f"{self._id}:{self._segredo}".encode()).decode()
        r = self._http.post("/v1/auth/token",
                            headers={"Authorization": f"Basic {credencial}"})
        r.raise_for_status()
        dados = r.json()
        self._token = dados["access_token"]
        # 🔑 Renove ANTES de expirar. A margem cobre o tempo de rede e o
        #    relógio dessincronizado entre as duas máquinas.
        margem = timedelta(seconds=1)
        self._expira = (datetime.now(timezone.utc)
                        + timedelta(seconds=dados["expires_in"]) - margem)
        self.renovacoes += 1

    def _cabecalho(self) -> dict[str, str]:
        if self._token is None or datetime.now(timezone.utc) >= self._expira:
            self._autenticar()
        return {"Authorization": f"Bearer {self._token}"}

    # ── chamadas ──
    def pedir(self, metodo: str, caminho: str, **kwargs) -> httpx.Response:
        return self._http.request(metodo, caminho, headers=self._cabecalho(), **kwargs)

    def cotar(self, peso_kg: float, cep_destino: str) -> dict:
        r = self.pedir("POST", "/v1/cotacoes",
                       json={"peso_kg": peso_kg, "cep_destino": cep_destino})
        r.raise_for_status()
        return r.json()

    def fechar(self):
        self._http.close()


veloz_cliente = ClienteVeloz(URL, "aurora-comercio", "s3nh4-da-aurora")
print("cotação:", veloz_cliente.cotar(2.4, "13010-000"))
print("cotação:", veloz_cliente.cotar(9.8, "01310-100"))
print(f"\nrenovações de token até agora: {veloz_cliente.renovacoes}")

In [ ]:
# 🎯 O token vale 4 segundos. Espere e chame de novo.
print(f"renovações antes : {veloz_cliente.renovacoes}")
print("aguardando o token expirar...")
time.sleep(4.2)

print("cotação:", veloz_cliente.cotar(1.0, "13010-000"))
print(f"renovações depois: {veloz_cliente.renovacoes}   ← renovou sozinho")

> 🎯 **Repare no que NÃO aconteceu:** nenhuma linha do código que chama `cotar()` sabe que existe token, expiração ou renovação. Isso está inteiramente encapsulado.
>
> Essa é a razão de existir uma classe cliente em vez de espalhar `httpx.get` pelo projeto. Quando a Veloz mudar de Basic para mTLS, você muda **um** arquivo.
>
> 🔴 **`cliente_id` e `segredo` vêm do ambiente**, nunca do código. É o mesmo `.env` do M06 — só que agora com credencial de terceiro, que costuma ser mais sensível ainda: ela dá acesso a um sistema que **não é seu**.

## 5. 🔴 Retry — e o que nunca se repete

In [ ]:
# Sem retry: uma falha transitória vira um erro para o usuário
ESTADO["chamadas"] = 0
ESTADO["falhas_restantes"] = 2         # falha 2 vezes, depois funciona

r = veloz_cliente.pedir("GET", "/v1/instavel")
mostrar(r, "sem retry (1ª tentativa)")
print("🔴 O usuário viu um erro. A próxima chamada teria funcionado.")

In [ ]:
# Com retry e espera exponencial
def com_retry(funcao, tentativas: int = 4, base: float = 0.2,
              repetir_status: tuple[int, ...] = (408, 425, 429, 500, 502, 503, 504)):
    """Repete falhas TRANSITÓRIAS, com espera exponencial e jitter.

    🔴 O jitter (aleatoriedade) não é frescura. Sem ele, mil clientes que
       falharam juntos voltam juntos — e derrubam de novo o serviço que
       estava se recuperando. É o "efeito manada".
    """
    ultima = None
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = funcao()
            if resposta.status_code not in repetir_status:
                return resposta
            ultima = httpx.HTTPStatusError(f"status {resposta.status_code}",
                                           request=resposta.request,
                                           response=resposta)
            motivo = f"status {resposta.status_code}"
        except httpx.RequestError as erro:
            ultima = erro
            motivo = type(erro).__name__

        if tentativa == tentativas:
            break
        espera = base * (2 ** (tentativa - 1))
        espera = random.uniform(espera * 0.5, espera)      # jitter
        print(f"   tentativa {tentativa} falhou ({motivo}) — esperando {espera:.2f}s")
        time.sleep(espera)

    raise ultima


ESTADO["chamadas"] = 0
ESTADO["falhas_restantes"] = 2

resposta = com_retry(lambda: veloz_cliente.pedir("GET", "/v1/instavel"))
mostrar(resposta, "com retry")

> 🎯 **Por que exponencial?** Se o serviço caiu, ele precisa de tempo. Repetir a cada 100 ms é bater na porta de alguém que está tentando se levantar — você piora o problema que quer contornar.
>
> `0,2s → 0,4s → 0,8s → 1,6s` dá espaço para a recuperação sem fazer o usuário esperar minutos.
>
> 🔴 **E o jitter?** Imagine 1.000 instâncias do seu serviço falhando no mesmo segundo. Sem jitter, as 1.000 voltam exatamente juntas, três vezes seguidas. Com jitter, elas se espalham.

In [ ]:
# 🔴 QUAIS ERROS SE REPETE?
#
# ⚠️ Marcadores ASCII, não emoji: `len("⏱️")` é 2 mas o terminal desenha
#    1 coluna, e a tabela sai torta. Você já viu isso no M03.
tabela = [
    ("400 Bad Request",       "[nao]", "O corpo está errado. Vai errar de novo."),
    ("401 Unauthorized",      "[nao]", "Renove o token primeiro — aí sim tente."),
    ("403 Forbidden",         "[nao]", "Sem permissão. Repetir não cria permissão."),
    ("404 Not Found",         "[nao]", "Não existe. Não vai passar a existir."),
    ("409 Conflict",          "[nao]", "Conflito de estado. Resolva, não repita."),
    ("422 Unprocessable",     "[nao]", "Dado inválido. Repetir é teimosia."),
    ("408 Request Timeout",   "[SIM]", "O servidor pediu que você refizesse."),
    ("429 Too Many Requests", "[SIM]", "Mas RESPEITE o Retry-After."),
    ("500 Internal Error",    "[SIM]", "Pode ter sido transitório."),
    ("502 Bad Gateway",       "[SIM]", "Proxy não achou o backend."),
    ("503 Unavailable",       "[SIM]", "Fora do ar temporariamente."),
    ("504 Gateway Timeout",   "[SIM]", "Backend demorou."),
    ("ConnectError",          "[SIM]", "Nem chegou a sair — seguro repetir."),
    ("ConnectTimeout",        "[SIM]", "Não abriu a conexão."),
    ("WriteTimeout",          "[ ? ]", "Parte do corpo pode ter chegado."),
    ("ReadTimeout",           "[ ? ]", "PODE TER SIDO PROCESSADO. Veja abaixo."),
]
print(f"{'Erro':<24}{'Repetir?':<11}Por quê")
print("─" * 78)
for erro, repete, motivo in tabela:
    print(f"{erro:<24}{repete:<11}{motivo}")
print("\n[nao] = nunca   ·   [SIM] = seguro   ·   [ ? ] = 🔴 só com idempotência")

> 🔴 **O `ReadTimeout` é a armadilha.**
>
> Você enviou `POST /cobranças` e não recebeu resposta. A cobrança aconteceu? **Você não tem como saber.** Repetir pode cobrar duas vezes. Não repetir pode não cobrar.
>
> **A saída é a idempotência.** Você manda uma chave única:
>
> ```
> Idempotency-Key: pedido-9042-tentativa-unica
> ```
>
> O servidor guarda o resultado associado à chave. Na segunda vez com a mesma chave, ele **devolve o resultado anterior** em vez de processar de novo. Repetir vira seguro.
>
> 🧭 **A regra prática:**
>
> | Método | Repetir sem idempotência? |
> |--------|---------------------------|
> | `GET`, `HEAD`, `OPTIONS` | ✅ sempre seguro — não mudam nada |
> | `PUT`, `DELETE` | ✅ idempotentes por definição |
> | **`POST`** | 🔴 **só com `Idempotency-Key`** |
>
> Você vai implementar o lado servidor disso na aula 07_02.

In [ ]:
# ⏱️ Limite de taxa: o servidor DIZ quanto esperar. Obedeça.
def respeitando_o_limite(cliente, caminho, chamadas=7):
    for i in range(1, chamadas + 1):
        r = cliente.get(caminho)
        if r.status_code == 429:
            espera = float(r.headers.get("Retry-After", 1))
            print(f"   {i}. ⏱️  429 — o servidor pediu {espera}s. Esperando.")
            time.sleep(espera + 0.05)
            r = cliente.get(caminho)
        restante = r.headers.get("X-RateLimit-Restante", "?")
        print(f"   {i}. ✅ {r.status_code}  (restam {restante} na janela)")


with httpx.Client(base_url=URL, timeout=5.0) as c:
    respeitando_o_limite(c, "/v1/limitado")

> ⏱️ **`Retry-After` é o serviço te dizendo exatamente o que fazer.** Ignorá-lo e usar o seu próprio backoff é, na melhor das hipóteses, ineficiente — e na pior, o caminho para ter a sua chave bloqueada.
>
> 💡 Muitos serviços mandam também `X-RateLimit-Remaining` e `X-RateLimit-Reset`. Um cliente educado desacelera **antes** de bater no limite, em vez de bater e recuar.

## 6. Disjuntor — parar de bater na porta

Retry resolve falha **transitória**. Mas se o serviço está fora há 10 minutos, cada requisição sua espera o timeout inteiro antes de falhar — e você acumula esperas de 5 segundos.

O **disjuntor** (*circuit breaker*) desiste rápido.

In [ ]:
class Disjuntor:
    """Três estados: fechado (normal) → aberto (desiste) → meio-aberto (testa).

    💭 A metáfora é o disjuntor elétrico: depois de N curtos, ele desarma
       e para de tentar. Passado um tempo, deixa passar UMA corrente de
       teste. Se ela passar bem, religa.
    """

    def __init__(self, limite_falhas: int = 3, espera_s: float = 1.0):
        self.limite = limite_falhas
        self.espera = espera_s
        self.falhas = 0
        self.aberto_ate = 0.0
        self.curtos = 0                # quantas chamadas foram barradas

    @property
    def estado(self) -> str:
        if time.monotonic() < self.aberto_ate:
            return "aberto"
        return "meio-aberto" if self.falhas >= self.limite else "fechado"

    def chamar(self, funcao):
        if self.estado == "aberto":
            self.curtos += 1
            # 🔑 Falha IMEDIATA — sem esperar timeout nenhum
            raise RuntimeError("disjuntor aberto: serviço considerado fora")
        try:
            resultado = funcao()
        except Exception:
            self.falhas += 1
            if self.falhas >= self.limite:
                self.aberto_ate = time.monotonic() + self.espera
            raise
        self.falhas = 0                # sucesso religa
        return resultado


disjuntor = Disjuntor(limite_falhas=3, espera_s=1.0)
morto = httpx.Client(base_url="http://127.0.0.1:1", timeout=0.3)

print("Serviço fora do ar, 8 tentativas:\n")
for i in range(1, 9):
    inicio = time.perf_counter()
    try:
        disjuntor.chamar(lambda: morto.get("/qualquer"))
        situacao = "✅ ok"
    except RuntimeError as erro:
        situacao = f"⚡ {erro}"
    except httpx.RequestError as erro:
        situacao = f"🔴 {type(erro).__name__}"
    ms = (time.perf_counter() - inicio) * 1000
    print(f"   {i}. [{disjuntor.estado:<11}] {ms:6.1f} ms  {situacao}")
    time.sleep(0.15)

print(f"\n{disjuntor.curtos} chamada(s) falharam em microssegundos em vez de esperar o timeout.")

> 🎯 **O ganho não é evitar o erro — é evitar a espera.**
>
> Com o disjuntor aberto, a falha é instantânea. Sem ele, cada chamada consome um worker por 300 ms (ou 5 segundos, com timeout realista) só para descobrir o que você já sabia.
>
> 💭 **Quando o disjuntor abre, o que a sua API responde?** Idealmente, não um `500`. Se a cotação de frete falhou, você pode devolver um valor estimado com um aviso, ou `503` com `Retry-After`. **Degradar é melhor do que quebrar** — o checkout continua funcionando, só sem o frete exato.
>
> 💡 Em produção, use uma biblioteca pronta (`tenacity` para retry, `purgatory` ou `pybreaker` para disjuntor). O valor de escrever à mão uma vez é entender o que elas fazem — e saber configurá-las.

## 7. Paginação

Ninguém devolve 4.000 registros de uma vez. Existem dois modelos, e eles não são equivalentes.

In [ ]:
# ── OFFSET: "me dê a página 3" ──
def paginar_por_offset(cliente, caminho, por_pagina=25):
    """Gerador: entrega item a item, buscando página a página.

    💡 Gerador (M04) é a estrutura certa aqui: quem consome escreve um
       `for` simples e não carrega 4.000 registros na memória.
    """
    pagina = 1
    while True:
        r = cliente.pedir("GET", caminho,
                          params={"pagina": pagina, "por_pagina": por_pagina})
        r.raise_for_status()
        dados = r.json()
        print(f"   → página {dados['pagina']}/{dados['total_paginas']} "
              f"({len(dados['itens'])} itens)")
        yield from dados["itens"]
        if pagina >= dados["total_paginas"]:
            return
        pagina += 1


print("Buscando todas as entregas por offset:")
todas = list(paginar_por_offset(veloz_cliente, "/v1/entregas", por_pagina=40))
print(f"\n✅ {len(todas)} entregas, sem nunca ter mais de 40 na memória")
print(f"   primeira: {todas[0]['codigo']}   última: {todas[-1]['codigo']}")

In [ ]:
# ── CURSOR: "me dê o que vem depois DESTE" ──
def paginar_por_cursor(cliente, caminho, limite=30):
    cursor = None
    pagina = 0
    while True:
        pagina += 1
        params = {"limite": limite}
        if cursor:
            params["cursor"] = cursor
        r = cliente.pedir("GET", caminho, params=params)
        r.raise_for_status()
        dados = r.json()
        print(f"   → lote {pagina} ({len(dados['itens'])} itens) "
              f"cursor={str(dados['proximo_cursor'])[:12]}")
        yield from dados["itens"]
        cursor = dados["proximo_cursor"]
        if not cursor:
            return


print("Buscando todos os eventos por cursor:")
eventos = list(paginar_por_cursor(veloz_cliente, "/v1/eventos", limite=50))
print(f"\n✅ {len(eventos)} eventos")

> 🎯 **Offset vs cursor — não é questão de gosto.**
>
> | | Offset | Cursor |
> |---|--------|--------|
> | Pular para a página 47 | ✅ sim | ❌ não |
> | Custo no banco | 🔴 `OFFSET 100000` é lento | ✅ constante |
> | Item inserido durante a leitura | 🔴 **você lê um registro duas vezes** | ✅ estável |
> | Item apagado durante a leitura | 🔴 **você pula um registro** | ✅ estável |
>
> 🔴 **O problema do offset é traiçoeiro.** Você lê a página 1 (itens 1–25). Alguém insere um registro no topo. Você lê a página 2 (itens 26–50) — mas o que era o item 25 virou o 26, e você o lê **de novo**.
>
> Numa importação noturna de 4.000 entregas, isso significa duplicatas silenciosas. Se a sua carga não for idempotente (M03), elas entram no banco.
>
> 🧭 **Regra:** offset para telas com "página 1 2 3"; **cursor para sincronização de dados**.

In [ ]:
# ⚠️ E a paginação que não termina?
def paginar_com_guarda(cliente, caminho, por_pagina=25, teto_paginas=500):
    """🔴 SEMPRE ponha um teto.

    Um bug no servidor que devolva sempre `total_paginas: 999999`, ou um
    cursor que aponta para si mesmo, transforma o seu `while True` num
    laço infinito que consome a cota da API e enche o disco de log.
    """
    vistos = set()
    pagina = 0
    while pagina < teto_paginas:
        pagina += 1
        r = cliente.pedir("GET", caminho,
                          params={"pagina": pagina, "por_pagina": por_pagina})
        r.raise_for_status()
        dados = r.json()
        if not dados["itens"]:
            return
        # 🔒 detecta repetição: sinal claro de bug do outro lado
        chave = dados["itens"][0]["codigo"]
        if chave in vistos:
            print(f"   ⚠️  página {pagina} repetiu o item {chave} — parando")
            return
        vistos.add(chave)
        yield from dados["itens"]
        if pagina >= dados["total_paginas"]:
            return
    print(f"   ⚠️  teto de {teto_paginas} páginas atingido — investigue")


total = sum(1 for _ in paginar_com_guarda(veloz_cliente, "/v1/entregas", 60))
print(f"✅ {total} entregas, com guarda contra laço infinito")

## 🔧 Prática guiada — o cliente da Veloz para o Atlas

Vamos juntar tudo num cliente que a Aurora poderia usar de verdade.

In [ ]:
class VelozResiliente(ClienteVeloz):
    """Cliente com timeouts, retry, disjuntor e paginação."""

    REPETIVEIS = (408, 425, 429, 500, 502, 503, 504)

    def __init__(self, *args, tentativas: int = 3, **kwargs):
        super().__init__(*args, **kwargs)
        self.tentativas = tentativas
        self.disjuntor = Disjuntor(limite_falhas=5, espera_s=2.0)
        self.metricas = {"chamadas": 0, "repeticoes": 0, "falhas": 0}

    def pedir(self, metodo: str, caminho: str, **kwargs) -> httpx.Response:
        """Uma chamada, com toda a resiliência aplicada."""
        def uma_vez():
            self.metricas["chamadas"] += 1
            return super(VelozResiliente, self).pedir(metodo, caminho, **kwargs)

        ultima = None
        for tentativa in range(1, self.tentativas + 1):
            try:
                resposta = self.disjuntor.chamar(uma_vez)
            except httpx.RequestError as erro:
                ultima = erro
                # 🔴 POST só se repete com chave de idempotência
                if metodo.upper() == "POST" and "Idempotency-Key" not in kwargs.get(
                        "headers", {}):
                    raise
            else:
                if resposta.status_code not in self.REPETIVEIS:
                    return resposta
                ultima = httpx.HTTPStatusError(
                    f"status {resposta.status_code}",
                    request=resposta.request, response=resposta)
                # ⏱️ obedece ao Retry-After quando ele existe
                if resposta.status_code == 429:
                    time.sleep(float(resposta.headers.get("Retry-After", 1)) + 0.05)
                    continue

            if tentativa == self.tentativas:
                break
            self.metricas["repeticoes"] += 1
            espera = 0.2 * (2 ** (tentativa - 1))
            time.sleep(random.uniform(espera * 0.5, espera))

        self.metricas["falhas"] += 1
        raise ultima

    def entregas(self, por_pagina: int = 50):
        """Todas as entregas, como um gerador."""
        yield from paginar_por_offset(self, "/v1/entregas", por_pagina)


atlas_veloz = VelozResiliente(URL, "aurora-comercio", "s3nh4-da-aurora")
print("✅ cliente resiliente pronto\n")

ESTADO["chamadas"] = 0
ESTADO["falhas_restantes"] = 2
r = atlas_veloz.pedir("GET", "/v1/instavel")
mostrar(r, "endpoint instável, via cliente resiliente")
print(f"métricas: {atlas_veloz.metricas}")

In [ ]:
# Uso realista: cotar frete para uma remessa
CARRINHO = [
    {"sku": "NB-DELL-15", "peso_kg": 2.4, "cep": "13010-000"},
    {"sku": "MO-LG-24UW", "peso_kg": 5.1, "cep": "01310-100"},
    {"sku": "PE-LOG-M170", "peso_kg": 0.3, "cep": "13480-000"},
    {"sku": "AR-KING-1TB", "peso_kg": 0.1, "cep": "SEM-CEP"},   # ⚠️ inválido
]

print(f"{'SKU':<14}{'peso':>8}  {'CEP':<12}{'frete':>11}  prazo")
print("─" * 58)
for item in CARRINHO:
    prefixo = f"{item['sku']:<14}{item['peso_kg']:>6.1f}kg  {item['cep']:<12}"
    try:
        cot = atlas_veloz.cotar(item["peso_kg"], item["cep"])
        print(f"{prefixo}R$ {cot['valor']:>8.2f}  {cot['prazo_dias']}d")
    except httpx.HTTPStatusError as erro:
        motivo = erro.response.json().get("detail", "?")
        print(f"{prefixo}{'--':>11}  [!] {motivo}")

print(f"\nmétricas: {atlas_veloz.metricas}")

In [ ]:
# Sincronização completa
print("Sincronizando todas as entregas:\n")
por_status: dict[str, int] = {}
por_cidade: dict[str, int] = {}

for entrega in atlas_veloz.entregas(por_pagina=50):
    por_status[entrega["status"]] = por_status.get(entrega["status"], 0) + 1
    por_cidade[entrega["cidade"]] = por_cidade.get(entrega["cidade"], 0) + 1

print(f"\n{'status':<14}qtd")
print("─" * 20)
for status, qtd in sorted(por_status.items(), key=lambda x: -x[1]):
    print(f"{status:<14}{qtd:>3}")

print(f"\n{'cidade':<14}qtd")
print("─" * 20)
for cidade, qtd in sorted(por_cidade.items(), key=lambda x: -x[1]):
    print(f"{cidade:<14}{qtd:>3}")

print(f"\nmétricas finais: {atlas_veloz.metricas}")

> 💭 **Compare com o script que a Aurora tinha:** um `requests.get` num laço, sem timeout, sem retry, sem paginação segura. Ele funcionava — nos dias em que tudo funcionava.
>
> O que você escreveu aqui funciona **nos outros dias também**. E isso é a diferença entre um script e um sistema.

In [ ]:
# Encerramento
atlas_veloz.fechar()
veloz_cliente.fechar()
cliente.close()
morto.close()
veloz_servico.parar()

## 📝 Exercícios

**E1.** Meça a diferença entre `httpx.get()` solto e `httpx.Client` com 100 requisições. Explique de onde vem a diferença.

**E2.** Configure os quatro timeouts do `httpx.Timeout` com valores diferentes e provoque cada um. (Dica: `PoolTimeout` exige `limits=httpx.Limits(max_connections=1)` e duas requisições simultâneas.)

**E3.** Escreva uma função que classifique uma exceção do httpx em "repetir", "não repetir" ou "depende", e justifique cada caso num comentário.

**E4.** Implemente um cliente com renovação de token que registre no log **toda** renovação. Prove que ele renova antes de expirar, não depois de falhar.

**E5.** 🔴 Implemente o retry com `tenacity` em vez de à mão. Compare o código e diga o que você ganhou e o que perdeu em clareza.

**E6.** Adicione ao retry um teto de tempo total (*deadline*): não importa quantas tentativas restem, nunca gaste mais de 10 segundos.

**E7.** Prove o efeito manada: 50 "clientes" repetindo sem jitter vs com jitter. Imprima o histograma de quando cada um voltou.

**E8.** Implemente o `Idempotency-Key` no **cliente**: gere a chave uma vez por operação lógica e reutilize-a em todas as tentativas daquela operação.

**E9.** Escreva um disjuntor com os três estados explícitos (`fechado`, `aberto`, `meio_aberto`) e teste a transição de volta para `fechado`.

**E10.** 🔴 Demonstre o problema do offset: pagine com `por_pagina=10` e insira um item no início da lista entre a página 1 e a 2. Mostre o registro duplicado.

**E11.** Escreva um `paginar()` genérico que funcione com offset **e** cursor, escolhendo pela presença de `proximo_cursor` na resposta.

**E12.** Implemente um limitador do **lado cliente**: no máximo N requisições por segundo, sem nunca receber um `429`.

**E13.** Adicione um cache de 60 segundos às cotações de frete (mesmo peso + mesmo CEP = mesma resposta). Meça as chamadas economizadas.

**E14.** 🔴 Faça a sincronização de entregas ser **retomável**: se ela morrer na página 40 de 100, a próxima execução continua de onde parou.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

## 📋 Cola de referência

```python
import httpx

# ═══ Client: um por serviço, reaproveitado 🎯 ═══
cliente = httpx.Client(
    base_url="https://api.servico.com",
    timeout=httpx.Timeout(5.0, connect=2.0),   # 🔴 NUNCA timeout=None
    headers={"User-Agent": "atlas-aurora/1.0 (voce@empresa.com)"},
    limits=httpx.Limits(max_connections=20, max_keepalive_connections=10),
)
cliente.close()          # ou use `with`

# ═══ Timeouts — quatro coisas diferentes ═══
httpx.Timeout(connect=2.0,   # abrir a conexão   → fora do ar
              read=5.0,      # esperar resposta  → servidor lento
              write=5.0,     # enviar o corpo    → upload
              pool=1.0)      # pegar do pool     → 🔴 problema SEU

# ═══ Erros: dois mundos ═══
r.raise_for_status()               # 4xx/5xx → HTTPStatusError
except httpx.HTTPStatusError:      # ✅ houve resposta
except httpx.RequestError:         # 🔴 não houve resposta
except httpx.HTTPError:            # os dois

#   HTTPError
#   ├── RequestError        ConnectError, ConnectTimeout, ReadTimeout,
#   │                       WriteTimeout, PoolTimeout, RemoteProtocolError
#   └── HTTPStatusError

# ═══ Retry — o que repetir 🔴 ═══
REPETIVEIS = (408, 425, 429, 500, 502, 503, 504)
# NÃO repetir: 400 401 403 404 409 422
# espera = base * 2**(n-1), com JITTER   ← senão, efeito manada
# ⏱️ 429 → obedeça ao cabeçalho Retry-After
# 🔴 POST só se repete com Idempotency-Key

# ═══ Disjuntor ═══
# fechado → (N falhas) → aberto → (espera) → meio-aberto → fechado
# Ganho: falhar em microssegundos em vez de esperar o timeout.

# ═══ Paginação ═══
# offset: ?pagina=3        pula páginas ✅ | 🔴 duplica/pula em lista viva
# cursor: ?cursor=abc      estável ✅      | não pula páginas
# 🧭 tela → offset · sincronização de dados → cursor
# 🔴 sempre com teto de páginas

# ═══ Produção ═══
# tenacity   → retry declarativo
# pybreaker  → disjuntor
# respx      → mock de httpx nos testes (aula 07_03)
```

## ✅ Checklist de saída

- [ ] Uso um `Client` por serviço, criado uma vez
- [ ] 🔴 **Todo cliente meu tem timeout explícito**
- [ ] Sei o que cada um dos quatro timeouts mede
- [ ] Entendo que `PoolTimeout` denuncia problema meu, não do outro
- [ ] Distingo "não houve resposta" de "a resposta foi ruim"
- [ ] Sei que o httpx **não** levanta exceção para 4xx/5xx sozinho
- [ ] Encapsulo autenticação de terceiro numa classe cliente
- [ ] Renovo o token **antes** de expirar, com margem
- [ ] 🔴 **Credencial de terceiro vem do ambiente**
- [ ] Sei quais status se repete e quais não
- [ ] Uso espera exponencial **com jitter**
- [ ] 🔴 **Não repito `POST` sem `Idempotency-Key`**
- [ ] Entendo por que o `ReadTimeout` é ambíguo
- [ ] Obedeço ao `Retry-After`
- [ ] Sei o que um disjuntor resolve que o retry não resolve
- [ ] Uso gerador para paginar
- [ ] Sei quando usar cursor em vez de offset
- [ ] 🔴 **Toda paginação minha tem teto**

---

### ➡️ Próxima aula

**`07_02_Design_e_Funcionalidades.ipynb`** — Agora do outro lado: webhooks (e como validar a assinatura), upload e download, tarefas em segundo plano, cache com Redis e WebSockets.